# Capítulo 5: Word Embeddings

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/caracena/apunte-analitica-textual/blob/main/capitulos/clase5-word-embeddings.ipynb)

## Objetivos de aprendizaje

- Comprender las limitaciones de las representaciones dispersas (BoW, TF-IDF).
- Entender el concepto de representaciones densas (embeddings).
- Implementar y utilizar Word2Vec para capturar relaciones semánticas.
- Visualizar y explorar espacios de embeddings.

## 5.1 De representaciones dispersas a densas

### Limitaciones de BoW y TF-IDF

- **Alta dimensionalidad**: Un vocabulario de 50.000 palabras genera vectores de 50.000 dimensiones.
- **Dispersión (*sparsity*)**: La mayoría de los valores son cero.
- **Sin semántica**: "rey" y "monarca" son dimensiones completamente independientes.
- **Sin contexto**: Una palabra siempre tiene la misma representación.

### La idea de los embeddings

Los **word embeddings** representan cada palabra como un vector **denso** de dimensión fija (típicamente 100-300 dimensiones), donde palabras semánticamente similares tienen vectores cercanos.

$$\text{"rey"} \rightarrow [0.25, -0.13, 0.87, ..., 0.42] \in \mathbb{R}^{300}$$

## 5.2 Word2Vec

**Word2Vec** (Mikolov et al., 2013) aprende embeddings a partir de grandes corpus de texto, basándose en la idea de que *"una palabra se conoce por la compañía que mantiene"*.

### Dos arquitecturas

1. **CBOW** (Continuous Bag of Words): Predice la palabra central a partir del contexto.
   - Entrada: palabras del contexto → Salida: palabra objetivo
   - Más rápido, mejor para palabras frecuentes

2. **Skip-gram**: Predice las palabras del contexto a partir de la palabra central.
   - Entrada: palabra objetivo → Salida: palabras del contexto
   - Mejor para palabras poco frecuentes y corpus pequeños

### Ejemplo conceptual de Skip-gram

Para la oración *"el gato se sentó en la alfombra"* con ventana de tamaño 2:

| Palabra objetivo | Contexto |
|-----------------|----------|
| gato | el, se, sentó |
| se | el, gato, sentó, en |
| sentó | gato, se, en, la |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from gensim.models import Word2Vec
from sklearn.decomposition import PCA

# Corpus de ejemplo para entrenar Word2Vec
oraciones = [
    ["el", "rey", "gobierna", "el", "reino", "con", "sabiduría"],
    ["la", "reina", "gobierna", "el", "reino", "con", "justicia"],
    ["el", "príncipe", "heredará", "el", "trono", "del", "rey"],
    ["la", "princesa", "heredará", "el", "trono", "de", "la", "reina"],
    ["el", "hombre", "trabaja", "en", "la", "ciudad"],
    ["la", "mujer", "trabaja", "en", "la", "ciudad"],
    ["el", "niño", "juega", "en", "el", "parque"],
    ["la", "niña", "juega", "en", "el", "parque"],
    ["el", "gato", "duerme", "en", "la", "alfombra"],
    ["el", "perro", "duerme", "en", "el", "sofá"],
    ["el", "rey", "y", "la", "reina", "viven", "en", "el", "palacio"],
    ["el", "hombre", "y", "la", "mujer", "viven", "en", "la", "casa"],
    ["el", "príncipe", "es", "hijo", "del", "rey"],
    ["la", "princesa", "es", "hija", "de", "la", "reina"],
    ["el", "gato", "y", "el", "perro", "son", "mascotas"],
    ["el", "niño", "y", "la", "niña", "son", "hermanos"],
    ["el", "sol", "brilla", "en", "el", "cielo"],
    ["la", "luna", "ilumina", "la", "noche"],
    ["los", "pájaros", "cantan", "en", "los", "árboles"],
    ["las", "flores", "crecen", "en", "el", "jardín"],
    ["el", "río", "fluye", "hacia", "el", "mar"],
    ["la", "montaña", "es", "alta", "y", "majestuosa"],
    ["un", "libro", "cuenta", "una", "historia"],
    ["una", "canción", "alegra", "el", "corazón"],
    ["el", "profesor", "enseña", "en", "la", "escuela"],
    ["el", "alumno", "aprende", "rápidamente"],
    ["el", "coche", "corre", "por", "la", "carretera"],
    ["la", "bicicleta", "es", "un", "medio", "de", "transporte", "saludable"],
    ["la", "comida", "es", "deliciosa", "y", "nutritiva"],
    ["el", "agua", "es", "esencial", "para", "la", "vida"],
    ["el", "tiempo", "vuela", "cuando", "te", "diviertes"],
    ["la", "amistad", "es", "un", "tesoro", "valioso"],
    ["el", "arte", "expresa", "sentimientos", "profundos"],
    ["la", "ciencia", "descubre", "nuevos", "conocimientos"],
    ["el", "doctor", "cura", "enfermedades"],
    ["la", "enfermera", "cuida", "a", "los", "pacientes"],
    ["el", "chef", "prepara", "platos", "exquisitos"],
    ["el", "camarero", "sirve", "la", "comida"],
    ["el", "músico", "toca", "un", "instrumento"],
    ["el", "bailarín", "ejecuta", "pasos", "elegantes"]
]

# Entrenar modelo Word2Vec
modelo_w2v = Word2Vec(
    sentences=oraciones,
    vector_size=50,     # Dimensión del embedding
    window=3,           # Tamaño de la ventana de contexto
    min_count=1,        # Frecuencia mínima
    sg=1,               # 1=Skip-gram, 0=CBOW
    epochs=200,         # Épocas de entrenamiento
    seed=42
)

print(f"Vocabulario: {len(modelo_w2v.wv)} palabras")
print(f"Dimensión de los embeddings: {modelo_w2v.wv.vector_size}")

In [ ]:
# Explorar el embedding de una palabra
vector_rey = modelo_w2v.wv['rey']
print(f"Vector de 'rey' (primeros 10 valores): {vector_rey[:10].round(3)}")
print(f"Norma del vector: {np.linalg.norm(vector_rey):.3f}")

In [ ]:
# Palabras más similares
print("Palabras más similares a 'rey':")
for palabra, sim in modelo_w2v.wv.most_similar('rey', topn=5):
    print(f"  {palabra}: {sim:.3f}")

print("\nPalabras más similares a 'gato':")
for palabra, sim in modelo_w2v.wv.most_similar('gato', topn=5):
    print(f"  {palabra}: {sim:.3f}")

## 5.3 Aritmética de palabras

Una propiedad fascinante de los word embeddings es que capturan relaciones semánticas mediante operaciones vectoriales:

$$\vec{\text{rey}} - \vec{\text{hombre}} + \vec{\text{mujer}} \approx \vec{\text{reina}}$$

Esto funciona porque los embeddings codifican dimensiones semánticas como género, realeza, etc.

In [ ]:
# Aritmética de palabras
# rey - hombre + mujer ≈ reina
resultado = modelo_w2v.wv.most_similar(
    positive=['rey', 'mujer'],
    negative=['hombre'],
    topn=3
)

print("rey - hombre + mujer =")
for palabra, sim in resultado:
    print(f"  {palabra}: {sim:.3f}")

## 5.4 Visualización de embeddings

In [ ]:
# Seleccionar palabras para visualizar
palabras_viz = ['rey', 'reina', 'príncipe', 'princesa',
                'hombre', 'mujer', 'niño', 'niña',
                'gato', 'perro']

# Obtener vectores
vectores = np.array([modelo_w2v.wv[p] for p in palabras_viz])

# Reducir a 2D con PCA
pca = PCA(n_components=2)
vectores_2d = pca.fit_transform(vectores)

# Graficar
fig, ax = plt.subplots(figsize=(10, 8))

# Colores por categoría
colores_map = {
    'rey': '#e74c3c', 'reina': '#e74c3c', 'príncipe': '#e74c3c', 'princesa': '#e74c3c',
    'hombre': '#3498db', 'mujer': '#3498db', 'niño': '#3498db', 'niña': '#3498db',
    'gato': '#2ecc71', 'perro': '#2ecc71'
}

for i, palabra in enumerate(palabras_viz):
    ax.scatter(vectores_2d[i, 0], vectores_2d[i, 1],
               c=colores_map[palabra], s=100, zorder=5)
    ax.annotate(palabra, (vectores_2d[i, 0], vectores_2d[i, 1]),
                fontsize=12, ha='center', va='bottom',
                fontweight='bold')

ax.set_title('Word Embeddings visualizados con PCA', fontsize=14)
ax.grid(True, alpha=0.3)

# Leyenda manual
from matplotlib.patches import Patch
leyenda = [Patch(color='#e74c3c', label='Realeza'),
           Patch(color='#3498db', label='Personas'),
           Patch(color='#2ecc71', label='Animales')]
ax.legend(handles=leyenda, fontsize=11)

plt.tight_layout()
plt.show()

## 5.5 Embeddings de documentos

Para representar un documento completo, una estrategia simple es promediar los embeddings de sus palabras.

In [ ]:
def documento_a_vector(documento, modelo):
    """Convierte un documento a un vector promediando los embeddings de sus palabras."""
    palabras = documento.lower().split()
    vectores = [modelo.wv[p] for p in palabras if p in modelo.wv]
    if not vectores:
        return np.zeros(modelo.wv.vector_size)
    return np.mean(vectores, axis=0)

# Ejemplo
doc1 = "el rey gobierna el reino"
doc2 = "la reina gobierna el reino"
doc3 = "el gato duerme en la alfombra"

vec1 = documento_a_vector(doc1, modelo_w2v)
vec2 = documento_a_vector(doc2, modelo_w2v)
vec3 = documento_a_vector(doc3, modelo_w2v)

from sklearn.metrics.pairwise import cosine_similarity

print("Similitud entre documentos:")
print(f"  '{doc1}' vs '{doc2}': {cosine_similarity([vec1], [vec2])[0][0]:.3f}")
print(f"  '{doc1}' vs '{doc3}': {cosine_similarity([vec1], [vec3])[0][0]:.3f}")

## 5.6 Modelos pre-entrenados

En la práctica, se suelen usar embeddings pre-entrenados en corpus masivos:

- https://crscardellino.net/SBWCE/
- https://github.com/dccuchile/spanish-word-embeddings#word2vec-embeddings-from-sbwc

In [ ]:
import gensim
from gensim.models import KeyedVectors
import numpy as np

# Ruta del archivo de embeddings pre-entrenados
model_path = '/content/SBW-vectors-300-min5.bin.gz'

print(f"Cargando el modelo Word2Vec español desde {model_path} (esto puede tardar)...")

# Cargar el modelo. El parámetro 'binary=True' es crucial para archivos .bin
try:
    modelo_espanol = KeyedVectors.load_word2vec_format(model_path, binary=True)
    print("Modelo español cargado correctamente.")

    print(f"Vocabulario del modelo: {len(modelo_espanol.index_to_key)} palabras")
    print(f"Dimensión de los embeddings: {modelo_espanol.vector_size}")

    # Ejemplos de uso con palabras en español
    print("\n--- Ejemplos de uso ---")

    # Palabras más similares
    palabra_ejemplo = 'rey'
    if palabra_ejemplo in modelo_espanol:
        print(f"\nPalabras más similares a '{palabra_ejemplo}':")
        for palabra, sim in modelo_espanol.most_similar(palabra_ejemplo, topn=5):
            print(f"  {palabra}: {sim:.3f}")
    else:
        print(f"\nLa palabra '{palabra_ejemplo}' no está en el vocabulario del modelo.")

    palabra_ejemplo_2 = 'ciencia'
    if palabra_ejemplo_2 in modelo_espanol:
        print(f"\nPalabras más similares a '{palabra_ejemplo_2}':")
        for palabra, sim in modelo_espanol.most_similar(palabra_ejemplo_2, topn=5):
            print(f"  {palabra}: {sim:.3f}")
    else:
        print(f"\nLa palabra '{palabra_ejemplo_2}' no está en el vocabulario del modelo.")

    # Aritmética de palabras (analogías)
    # rey - hombre + mujer = reina
    positive_words = ['rey', 'mujer']
    negative_words = ['hombre']
    
    # Verificar que todas las palabras estén en el vocabulario
    if all(p in modelo_espanol for p in positive_words + negative_words):
        resultado_analogia = modelo_espanol.most_similar(
            positive=positive_words,
            negative=negative_words,
            topn=3
        )
        print(f"\nAnalogía: {' + '.join(positive_words)} - {' - '.join(negative_words)} =")
        for palabra, sim in resultado_analogia:
            print(f"  {palabra}: {sim:.3f}")
    else:
        missing_words = [p for p in positive_words + negative_words if p not in modelo_espanol]
        print(f"\nNo se pudo realizar la analogía. Faltan palabras en el vocabulario: {missing_words}")

except Exception as e:
    print(f"Error al cargar o usar el modelo: {e}")

## 5.7 Clasificación usando Word embeddings

Revisemos el ejercicio de clasificación que hicimos en el capítulo 4.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline

# Datos de ejemplo: reseñas de productos con sentimiento
textos = [
    # Positivas
    "Excelente producto, superó mis expectativas",
    "Muy buena calidad, lo recomiendo totalmente",
    "Increíble relación calidad precio, estoy encantado",
    "El mejor producto que he comprado en mucho tiempo",
    "Funciona perfecto, llegó antes de lo esperado",
    "Me encantó, muy fácil de usar y bonito diseño",
    "Gran compra, todos en casa están felices",
    "Producto de primera calidad, totalmente satisfecho",
    "Fantástico, cumple con todo lo prometido",
    "Muy contento con la compra, lo volvería a comprar",
    # Negativas
    "Pésima calidad, se rompió al segundo día",
    "No funciona como dice la descripción, decepcionado",
    "Muy malo, no lo recomiendo para nada",
    "El producto llegó dañado y el servicio es terrible",
    "Una pérdida de dinero, no vale lo que cuesta",
    "Horrible experiencia de compra, nunca más",
    "El peor producto que he probado, no sirve",
    "Mala calidad, el material es muy frágil",
    "No cumple con las expectativas, muy decepcionante",
    "Terrible, tuve que devolverlo inmediatamente"
]

etiquetas = ["positivo"] * 10 + ["negativo"] * 10

# Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(
    textos, etiquetas, test_size=0.3, random_state=42, stratify=etiquetas
)

print(f"Entrenamiento: {len(X_train)} documentos")
print(f"Prueba: {len(X_test)} documentos")

Usemos redes neuronales y vectorización TF-IDF

In [ ]:
# 1. Initialize and fit TfidfVectorizer on training data
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train)
X_test_tfidf = tfidf_vectorizer.transform(X_test)

# 2. Train MLP Classifier directly with TF-IDF features
clf_mlp_tfidf = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
clf_mlp_tfidf.fit(X_train_tfidf, y_train)

# 3. Predict and evaluate
y_pred_mlp_tfidf = clf_mlp_tfidf.predict(X_test_tfidf)

print("=== Red Neuronal (MLP) con TF-IDF (sin Pipeline) ===")
print(classification_report(y_test, y_pred_mlp_tfidf))

Ahora solo cambiemos la vectorización usando Word Embeddings

In [ ]:
def documento_a_vector(documento, modelo):
    palabras = documento.lower().split()
    vectores = [modelo[p] for p in palabras if p in modelo]
    if not vectores:
        return np.zeros(modelo.vector_size)
    return np.mean(vectores, axis=0)

# 1. Convert training and test data to Word Embeddings
X_train_we = np.array([documento_a_vector(doc, modelo_espanol) for doc in X_train])
X_test_we = np.array([documento_a_vector(doc, modelo_espanol) for doc in X_test])

# 2. Train MLP Classifier directly with Word Embeddings
clf_mlp_we = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
clf_mlp_we.fit(X_train_we, y_train)

# 3. Predict and evaluate
y_pred_mlp_we = clf_mlp_we.predict(X_test_we)

print("=== Red Neuronal (MLP) con Word Embeddings (sin Pipeline/Clase Custom) ===")
print(classification_report(y_test, y_pred_mlp_we))

## Resumen

En este capítulo aprendimos:

- **Word Embeddings**: Representaciones densas que capturan significado semántico.
- **Word2Vec**: Aprende embeddings usando CBOW o Skip-gram a partir de contextos.
- **Aritmética de palabras**: Los embeddings codifican relaciones semánticas como analogías.
- **Embeddings de documentos**: Se pueden obtener promediando los vectores de las palabras.
- **Modelos pre-entrenados**: GloVe, FastText ofrecen embeddings de alta calidad.

En el próximo capítulo daremos el salto a los **Transformers** y **LLMs**, que llevan las representaciones contextuales al siguiente nivel.